In [1]:
from random import Random

In [2]:
from torchvision import datasets, transforms

In [3]:
class Partition(object):
    """Dataset-like object, but only access a subset of it."""

    def __init__(self, data, index):
        self.data = data
        self.index = index

    def __len__(self):
        return len(self.index)

    def __getitem__(self, index):
        data_idx = self.index[index]
        return self.data[data_idx]

In [4]:
class DataPartitioner(object):
    """Partitions a dataset into different chuncks."""

    def __init__(self, data, sizes=[0.7, 0.2, 0.1], seed=1234):
        self.data = data
        # print("DataPartitioner __init__ self.data:", self.data)

        self.partitions = []

        rng = Random()
        rng.seed(seed)

        data_len = len(data)
        print("DataPartitioner __init__ data_len:", data_len)
        # DataPartitioner __init__ data_len: 60000

        indexes = [x for x in range(0, data_len)]
        # print("DataPartitioner __init__ indexes:", indexes)

        rng.shuffle(indexes)
        # print("DataPartitioner __init__ indexes:", indexes)

        for frac in sizes:
            print("DataPartitioner __init__ frac:", frac)

            part_len = int(frac * data_len)
            print("DataPartitioner __init__ part_len:", part_len)

            self.partitions.append(indexes[0:part_len])
            indexes = indexes[part_len:]

        print("DataPartitioner __init__ self.partitions:", self.partitions)

    def use(self, partition):
        return Partition(self.data, self.partitions[partition])

In [5]:
dataset = datasets.MNIST(
    "./data",
    train=True,
    download=True,
    transform=transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
    ),
)

In [6]:
size = 2
bsz = 128 // size
bsz

64

In [7]:
partition_sizes = [1.0 / size for _ in range(size)]
partition_sizes

[0.5, 0.5]

In [8]:
partition = DataPartitioner(dataset, partition_sizes)

DataPartitioner __init__ data_len: 60000
DataPartitioner __init__ frac: 0.5
DataPartitioner __init__ part_len: 30000
DataPartitioner __init__ frac: 0.5
DataPartitioner __init__ part_len: 30000
DataPartitioner __init__ self.partitions: [[33712, 36159, 45528, 25849, 53734, 40353, 32418, 55928, 38521, 21823, 15630, 53486, 42658, 4899, 9832, 18628, 39371, 39211, 58610, 37514, 2010, 49619, 23758, 23619, 14927, 27206, 5720, 37747, 53946, 44549, 19923, 19453, 39527, 57229, 21039, 17713, 28170, 26903, 34855, 35973, 44918, 12467, 39291, 48137, 47075, 5488, 42361, 46237, 56056, 59642, 3739, 45, 32227, 58636, 25345, 23706, 27262, 54234, 1697, 37446, 19689, 58648, 13466, 35799, 29764, 22959, 50576, 57953, 30052, 4618, 58910, 48485, 36818, 54092, 50844, 1457, 5523, 29232, 15462, 17221, 47233, 18479, 46644, 716, 42750, 25747, 32273, 29125, 42964, 43518, 33134, 38102, 40222, 35546, 31948, 14588, 38429, 57019, 7, 20259, 59764, 55369, 42876, 51570, 32478, 34657, 46071, 56096, 3091, 30298, 10395, 32856,

In [9]:
partition = partition.use(1)
partition